# TinyAya MapHallu — Analysis Notebook

Evaluation of four TinyAya model variants (global, earth, fire, water) across
hallucination rate, cross-model disagreement, answer match rate, and prompt sensitivity.

In [ ]:
import re
import sys
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

rc_defaults = {
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "lines.linewidth": 1.8,
    "lines.markersize": 6,
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "savefig.bbox_inches": "tight",
}
plt.rcParams.update(rc_defaults)

MODEL_PALETTE = {
    "global": "#5B8DB8",
    "fire": "#E45756",
    "earth": "#E07B54",
    "water": "#6BAE8E",
}
MODEL_ORDER = ["global", "earth", "fire", "water"]
LANG_ORDER = ["en", "ar", "de", "ru", "th", "zh"]


def short_model(name: str) -> str:
    return name.replace("tiny-aya-", "")


DATA_ROOT = Path("../run_experiments/output")
SKIP_DIRS = {"water_base_20260322_051223"}
OUTPUT_DIR = Path("plots")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
sys.path.insert(0, ".")
from load_data import load_all

mkqa_base, xnli_base, mkqa_pss = load_all(DATA_ROOT, skip=SKIP_DIRS)

for df in [mkqa_base, xnli_base, mkqa_pss]:
    df["model"] = df["model"].map(short_model)

print("mkqa_base:", mkqa_base.shape)
print(mkqa_base["model"].value_counts())
print(mkqa_base["language"].value_counts())
print()
print("xnli_base:", xnli_base.shape)
print(xnli_base["model"].value_counts())
print(xnli_base["language"].value_counts())
print()
print("mkqa_pss:", mkqa_pss.shape)
print(mkqa_pss["model"].value_counts())
print(mkqa_pss["language"].value_counts())

## Hallucination Rate

In [ ]:
# Hallucination Rate by model (rep=0 to avoid inflation)
hr_df = mkqa_base[mkqa_base["rep"] == 0].copy()
hr_model = (
    hr_df.groupby("model")["is_correct"]
    .mean()
    .reindex(MODEL_ORDER)
    .rename("accuracy")
    .reset_index()
)
hr_model["hallucination_rate"] = (1 - hr_model["accuracy"]) * 100

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    data=hr_model, x="model", y="hallucination_rate",
    palette=MODEL_PALETTE, order=MODEL_ORDER, ax=ax,
)
ax.set_title("Hallucination Rate by Model")
ax.set_ylabel("Hallucination Rate (%)")
ax.set_xlabel("Model")
for bar in ax.patches:
    ax.annotate(f"{bar.get_height():.1f}%",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "hr_by_model.png")
plt.show()

In [ ]:
# Hallucination Rate by model × language
hr_lang = (
    hr_df.groupby(["language", "model"])["is_correct"]
    .mean()
    .rename("accuracy")
    .reset_index()
)
hr_lang["hallucination_rate"] = (1 - hr_lang["accuracy"]) * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=hr_lang, x="language", y="hallucination_rate", hue="model",
    palette=MODEL_PALETTE, hue_order=MODEL_ORDER, order=LANG_ORDER, ax=ax,
)
ax.set_title("Hallucination Rate by Language × Model")
ax.set_ylabel("Hallucination Rate (%)")
ax.set_xlabel("Language")
ax.legend(title="Model")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "hr_by_lang_model.png")
plt.show()

## CMDR — Cross-Model Disagreement Rate

In [ ]:
# CMDR across languages (per model): label disagreement between language pairs
xnli_r0 = xnli_base[xnli_base["rep"] == 0].copy()
XNLI_LANGS = sorted(xnli_r0["language"].unique())

fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(5 * len(MODEL_ORDER), 4.5), sharey=True)

for ax, model in zip(axes, MODEL_ORDER):
    mdf = xnli_r0[xnli_r0["model"] == model]
    pivot = mdf.pivot_table(index="sample_id", columns="language", values="parsed_label", aggfunc="first")
    n_langs = len(XNLI_LANGS)
    disagree = pd.DataFrame(np.zeros((n_langs, n_langs)), index=XNLI_LANGS, columns=XNLI_LANGS)

    for l1, l2 in combinations(XNLI_LANGS, 2):
        if l1 in pivot.columns and l2 in pivot.columns:
            mask = pivot[l1].notna() & pivot[l2].notna()
            rate = (pivot.loc[mask, l1] != pivot.loc[mask, l2]).mean()
            disagree.loc[l1, l2] = rate
            disagree.loc[l2, l1] = rate

    sns.heatmap(disagree.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                vmin=0, vmax=0.5, ax=ax, cbar=model == MODEL_ORDER[-1],
                square=True)
    ax.set_title(model)

fig.suptitle("CMDR: Label Disagreement Across Languages (per model)", fontsize=15, y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "cmdr_lang_heatmaps.png")
plt.show()

In [ ]:
# CMDR across models: label disagreement between model pairs per language
rows = []
for lang in XNLI_LANGS:
    ldf = xnli_r0[xnli_r0["language"] == lang]
    pivot = ldf.pivot_table(index="sample_id", columns="model", values="parsed_label", aggfunc="first")
    for m1, m2 in combinations(MODEL_ORDER, 2):
        if m1 in pivot.columns and m2 in pivot.columns:
            mask = pivot[m1].notna() & pivot[m2].notna()
            rate = (pivot.loc[mask, m1] != pivot.loc[mask, m2]).mean()
            rows.append({"language": lang, "pair": f"{m1}–{m2}", "disagreement": rate})

cmdr_models = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=cmdr_models, x="language", y="disagreement", hue="pair",
    order=XNLI_LANGS, ax=ax,
)
ax.set_title("CMDR: Label Disagreement Across Models (per language)")
ax.set_ylabel("Disagreement Rate")
ax.set_xlabel("Language")
ax.legend(title="Model Pair", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "cmdr_model_pairs.png")
plt.show()

## AMR, SCS & CLC

In [ ]:
# AMR (Answer Match Rate)
def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def answer_match(response, gold_answers):
    """Score response against a list of gold answer strings."""
    if not gold_answers:
        return None
    if not response:
        return 0.0
    r = normalize_text(response)
    best = 0.0
    for g_text in gold_answers:
        g = normalize_text(g_text)
        if not g:
            continue
        if g in r:
            return 1.0
        r_tokens = set(r.split())
        g_tokens = set(g.split())
        overlap = len(r_tokens & g_tokens)
        score = overlap / len(g_tokens) if g_tokens else 0.0
        best = max(best, score)
    return round(best, 4)


amr_df = mkqa_base[mkqa_base["rep"] == 0].copy()
amr_df["amr"] = amr_df.apply(
    lambda row: answer_match(row["response_text"], row["gold_answers"]), axis=1
)
amr_df = amr_df.dropna(subset=["amr"])

amr_lang = amr_df.groupby(["language", "model"])["amr"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=amr_lang, x="language", y="amr", hue="model",
    palette=MODEL_PALETTE, hue_order=MODEL_ORDER, order=LANG_ORDER, ax=ax,
)
ax.set_title("Answer Match Rate (AMR) by Language × Model")
ax.set_ylabel("AMR")
ax.set_xlabel("Language")
ax.legend(title="Model")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "amr_by_lang_model.png")
plt.show()

In [ ]:
# AMR Heatmap
amr_pivot = amr_lang.pivot(index="model", columns="language", values="amr")
amr_pivot = amr_pivot.reindex(index=MODEL_ORDER, columns=LANG_ORDER)

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(
    amr_pivot, annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0, vmax=1, ax=ax, linewidths=0.5,
)
ax.set_title("AMR Heatmap (Model × Language)")
ax.set_ylabel("Model")
ax.set_xlabel("Language")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "amr_heatmap.png")
plt.show()

### SCS — Semantic Consistency Score

SCS computation requires sentence-transformers embeddings — run separately.

The approach:
1. Group mkqa_base by `(sample_id, model)` with rep=0.
2. Embed `response_text` for each language using a multilingual sentence-transformer.
3. Compute pairwise cosine similarity across all language pairs.
4. SCS = mean cosine similarity per (sample_id, model).

In [ ]:
# SCS placeholder — uncomment when embeddings are available
#
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
#
# scs_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#
# scs_df = mkqa_base[mkqa_base["rep"] == 0].copy()
# scs_rows = []
# for (sid, model), grp in scs_df.groupby(["sample_id", "model"]):
#     texts = grp.set_index("language")["response_text"].dropna()
#     if len(texts) < 2:
#         continue
#     embs = scs_model.encode(texts.tolist())
#     sim = cosine_similarity(embs)
#     triu = sim[np.triu_indices_from(sim, k=1)]
#     scs_rows.append({"sample_id": sid, "model": model, "scs": triu.mean()})
#
# scs_results = pd.DataFrame(scs_rows)
# print(scs_results.groupby("model")["scs"].mean())

print("SCS: skipped (requires embedding computation)")

## PSS — Prompt Sensitivity Score

In [ ]:
# PSS: Accuracy by variant type
VARIANT_ORDER = ["base", "paraphrase", "short", "instruction", "context"]
pss_r0 = mkqa_pss[mkqa_pss["rep"] == 0].copy()

pss_acc = (
    pss_r0.groupby(["variant_type", "model"])["is_correct"]
    .mean()
    .rename("accuracy")
    .reset_index()
)
pss_acc["accuracy_pct"] = pss_acc["accuracy"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=pss_acc, x="variant_type", y="accuracy_pct", hue="model",
    palette=MODEL_PALETTE, hue_order=MODEL_ORDER, order=VARIANT_ORDER, ax=ax,
)
ax.set_title("PSS: Accuracy by Prompt Variant")
ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Variant Type")
ax.legend(title="Model")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "pss_accuracy_by_variant.png")
plt.show()

In [ ]:
# PSS: Accuracy drop from base variant
base_acc = (
    pss_r0[pss_r0["variant_type"] == "base"]
    .groupby(["model", "language"])["is_correct"]
    .mean()
    .rename("base_accuracy")
)
all_acc = (
    pss_r0.groupby(["model", "language", "variant_type"])["is_correct"]
    .mean()
    .rename("accuracy")
    .reset_index()
)
all_acc = all_acc.join(base_acc, on=["model", "language"])
all_acc["drop"] = (all_acc["base_accuracy"] - all_acc["accuracy"]) * 100

# Average drop across languages
drop_summary = (
    all_acc[all_acc["variant_type"] != "base"]
    .groupby(["model", "variant_type"])["drop"]
    .mean()
    .reset_index()
)
drop_pivot = drop_summary.pivot(index="model", columns="variant_type", values="drop")
drop_pivot = drop_pivot.reindex(
    index=MODEL_ORDER,
    columns=[v for v in VARIANT_ORDER if v != "base"],
)

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.heatmap(
    drop_pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
    center=0, ax=ax, linewidths=0.5,
)
ax.set_title("PSS: Accuracy Drop from Base Variant (pp)")
ax.set_ylabel("Model")
ax.set_xlabel("Variant Type")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "pss_accuracy_drop_heatmap.png")
plt.show()